In [1]:
# set the name of the (GNSS) RINEX file to use
rinex_fn = "../../../data/DYNG00GRC_R_20240010000_01D_30S_MO.rnx"

* ένα sp3[cd] αρχείο, με τροχιές δουρφόρων GPS που να καλύπτει το διάστημα των παρατηρήσεων του αρχείου RINEX

In [2]:
# set the name of the Sp3 file for the same satellite, covering the same day
sp3_fn = "../../../data/COD0MGXFIN_20240010000_01D_05M_ORB.SP3"

* αρχείο μετεωρολογικών δεδομένων gpt3 (Global Pressure Temperature Model)

In [3]:
# gpt35_grid = "../../data/gpt3_5.grd"
#
# https://vmf.geo.tuwien.ac.at/trop_products/GNSS/VMF3/VMF3_OP/yearly/y2024.vmf3_g
#
vmf3_data = "../../../data/y2024.vmf3_g"

#
# http://ftp.aiub.unibe.ch/CODE/2024/
#
dcb_data = "../../../data/COD0MGXFIN_20240010000_01D_01D_OSB.BIA"

#
# https://files.igs.org/pub/station/general/
#
atx_data = "../../../data/igs20.atx"

* τις παρακάτω βιβλιοθήκες της Python

In [4]:
from dsoclasses.orbits.interpolator import Sp3Interpolator
from dsoclasses.geodesy import transformations
from dsoclasses.gnss import systems as gs
from dsoclasses.gnss import algorithms as alg
from dsoclasses.rinex.gnss.rinex import GnssRinex, fetch, fetchv
from dsoclasses.troposphere.vmf3 import SiteVmf3
from dsoclasses.time.pyattotime import at2pt, fsec2asec
import dsoclasses.gnss.atx as atx
from dsoclasses.gnss.biasx import Bsinex
from dsoclasses.gnss.atx import Atx
import attotime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from scipy.spatial.transform import Rotation as nRot

* συναρτήσεις/μεταβλητές που δημιουργήσαμε στο gps_code_analysis.ipynb. Θα υπολογίσουμε επίσης την διόρθωση λόγω σχετικότητας, λόγω της (σχετικήες) ταχύτητας δορυφόρου-δέκτη: $\delta _{rel} = 2 \cdot \frac{\vec{r}\cdot\vec{v}}{c}$

In [5]:
# Γεωμετρική απόσταση δορυφόρου-δέκτη
geometric_range = alg.geometric_range

# Συν/νες δορυφόρου την εποχή εκπομπής
sat_at_emission_time = alg.sat_at_emission_time

# Προσεγγιστικός υπολογισμός ταχύτητας δορυφόρου, από διαδοχικές θέσεις
def sat_vel(t, interpolator, sat, dsec=.5):
    xn1, yn1, zn1, _ = interpolator.sat_at(sat, t+attotime.attotimedelta(milliseconds=dsec*1e3))
    xc, yc, zc, _ = interpolator.sat_at(sat, t)
    xn2, yn2, zn2, _ = interpolator.sat_at(sat, t+attotime.attotimedelta(milliseconds=2*dsec*1e3))
    vel1 = (np.array((xn1, yn1, zn1)) - np.array((xc, yc, zc))) / dsec
    vel2 = (np.array((xn2, yn2, zn2)) - np.array((xc, yc, zc))) / dsec / 2
    return (2*vel1 + 1*vel2) / 3., np.array((xc, yc, zc))

# Διόρθωση σχετικότητας
def relativity(t_reception, dt, interpolator, sat, dsec=.5):
    t_emission = t_reception - fsec2asec(dt)
    vel, pos = sat_vel(t_emission, interpolator, sat, dsec)
    return 2. * np.inner(pos, vel) / gs.C

def sagnac(rsat, rsta):
    return (gs.OmegaEarth / gs.C) * (rsat[0]*rsta[1] - rsat[1]*rsta[0])

Για τον υπολογισμό της τροποσφαιρικής επίδρασης με το μοντέλο GPT/GMF θα χρησιμοποιήσουμε την συνάρτηση: `tropo_delay(rsta, t, el, gpt3_grid)`, όπου το `gpt3_grid` είναι το αρχείο των μετεωρολογικών δεδομένων.

In [6]:
# Παρεμβολή για τις τροχιές
intrp = intrp = Sp3Interpolator.from_sp3(sp3_fn, ['G'], interval_in_sec=1800, min_data_pts=12, itype='Barycentric')
# Αρχικοποίηση του αντικειμένου GnssRinex
rnx = GnssRinex(rinex_fn)
# προσεγγιστικές συν/νες δέκτη (από RINEX)
rsta = np.array(rnx.approx_cartesian())
# Μετατροπή σε ελλειψοειδείς συν/νες (δέκτης)
lat, lon, hgt = transformations.car2ell(*rsta)
# Tropo
vmf = SiteVmf3(vmf3_data, [rnx.marker_name])
# DCB
bsnx = Bsinex(dcb_data)
# Atx
atx = Atx(atx_data)

In [7]:
# Δεν θα χρησιμοποιήσοουμε τον δορυφόρο 27, λόγω πολύ μεγάλων υπολοίπων
EXCLUDE_SATELLITES = []

# Γωνία αποκοπής· θα χρησιμοποιήσουμε μόνο παρατηρήσεις σε γωνία ύψους μεγαλύτερη από αυτή
ELEVATION_CUT_OFF  = np.radians(3)

# Για ευκολία, μπορούμε να χρησιμοποιήσουμε μόνο παρατηρήσεις που απέχουν μεταξύ τους κάποια δευτερόλεπτα. Έτσι, 
# θα έχουμε μικρότερο αριθμό παρατηρήσεων προς επεξεργασία
EVERY_SEC          = 180.

f1 = gs.GPS_L1_FREQ # MHz
f2 = gs.GPS_L2_FREQ # aka E5a MHz
fac = f1*f1 - f2*f2
SAT_SYS = 'G'
FREQ1 = 'L1'
FREQ2 = 'L2'
lambda_wl = gs.C / (f1 - f2)

In [20]:
def tec(l1, l2, DN12=0., bs=0., br=0.):
    lambda1 = gs.C / f1
    lambda2 = gs.C / f2
    gamma   = f1*f1 / f2*f2
    return f1*f1*((lambda1*l1 - lambda2*l2) - DN12 - bs - br) / (40.3e16*(gamma-1.))

MAX_TECR_SEC = 180.
def smoothed_tec(t, tecr_history):
    km1 = max([tr[1] for tr in tecr_history])
    tecr_km1 = np.mean([tr[0] for tr in tecr_history if (km1-tr[1]).total_nanoseconds() <= MAX_TECR_SEC*1e9])
    skm1 = np.std([tr[0] for tr in tecr_history if (km1-tr[1]).total_nanoseconds() <= MAX_TECR_SEC*1e9])
    km2 = max([tr[1] for tr in tecr_history if tr[1]<km1])
    tecr_km2 = np.mean([tr[0] for tr in tecr_history if (km2-tr[1]).total_nanoseconds() <= MAX_TECR_SEC*1e9])
    skm2 = np.std([tr[0] for tr in tecr_history if (km2-tr[1]).total_nanoseconds() <= MAX_TECR_SEC*1e9])
    tecr_km1_dot = (tecr_km1 - tecr_km2) / ((km1-km2).total_nanoseconds()*1e-9)
    tecr = tecr_km1 + tecr_km1_dot * ((t-km1).total_nanoseconds()*1e-9)
    stecr = np.sqrt(skm1*skm1+((t-km1).total_nanoseconds()*1e-9)*skm2*skm2)
    return tecr, stecr
    
def clean_tecr_hist(t, tecr_history)
    ## Remove all entries of a passed-in satdct[sat]['TECR_hist'], for which the 
    ## reference epoch is larger than MAX_TECR_SEC away from t
        
satdct ={}
for block in rnx:
    t = block.t()
    for sat, obs in block.filter_satellite_system("G", False):
        try:
            p1, ctype1 = fetch(obs, 'C1W', 'C1C', 'C1X')
            p2, ctype2 = fetch(obs, 'C5X', 'C5I', 'C5Q')
            cbias1 = bsnx.sat_bias(sat)[ctype1]
            cbias2 = bsnx.sat_bias(sat)[ctype2]
            p1 = p1['value'] - cbias1 * 1e-9 * gs.C
            p2 = p2['value'] - cbias2 * 1e-9 * gs.C

            l1 = fetchv(obs, 'L1C', 'L1L', 'L1P', 'L1Y')
            l2 = fetchv(obs, 'L2C', 'L2L', 'L2P', 'L2Y')

            # MWWL --------------------------------
            # Eq. 6: Wide-Lane ambiguity at epoch t
            mwwl_cycle_slip = False
            Nwl = l1 - l2 - (f1*p1 + f2*p2)/(lambda_wl*(f1+f2))
            if sat not in satdct:
                satdct[sat] = {'k': 1, 'Nwl_prev': 0e0, 'Nwl_next': Nwl 'Nmean': Nwl, 'Naux': Nwl*Nwl, 'Nstd': 0e0, 
                               'TEC_prev': (0.,t), 'TEC_next': (0.,t), 'TECR_hist':[]}
            else:
                satdct[sat]['k'] += 1
                satdct[sat]['Nmean'] += (Nwl - satdct[sat]['Nmean'])/satdct[sat]['k']
                satdct[sat]['Naux']  += (Nwl*Nwl - satdct[sat]['Naux'])/satdct[sat]['k']
                satdct[sat]['Nstd']   = np.sqrt(satdct[sat]['Naux'] - satdct[sat]['Nmean']*satdct[sat]['Nmean'])
                satdct[sat]['Nwl_prev'] = satdct[sat]['Nwl_next']
                satdct[sat]['Nwl_next'] = Nwl
                # check for cycle slip:
                if np.abs(satdct[sat]['Nwl_next']-satdct[sat]['Nwl_prev']) > 4. * satdct[sat]['Nstd']:
                    mwwl_cycle_slip = True

            # TEC -----------------------------
            tecr_cycle_slip = False
            # compute TEC(k), using Eq. 10
            tec_meas = tec(l1,l2,0.,0.,0.)
            if satdct[sat]['k'] == 1:
                satdct[sat]['TEC_next'] =  (tec_meas, t)
            elif satdct[sat]['k'] > 1:
                satdct[sat]['TEC_prev'] = satdct[sat]['TEC_next']
                satdct[sat]['TEC_next'] = (tec_meas, t)
                # compute TECR(k) from Eq. 11
                tecr_meas = (satdct[sat]['TEC_next'][0] - satdct[sat]['TEC_prev'][0]) / ((satdct[sat]['TEC_next'][1] - satdct[sat]['TEC_prev'][1]).total_nanoseconds() * 1e-9)
                if satdct[sat]['k'] > 3:
                    # average TECR(k)
                    tecr_13, tecr_std = smoothed_tec(t, satdct[sat]['TECR_hist'])
                    if np.abs(tecr_13-tecr_meas) > 4. * tecr_std:
                        tecr_cycle_slip = True
                satdct[sat]['TECR_hist'].append((tecr_meas, t))
                clean_tecr_hist(t, satdct[sat]['TECR_hist'])
                                
        except Exception as e:
            print(f"{e}")

Number of obsevrations: 3708
